#### Tool Calling

In [1]:
from langgraph.prebuilt import ToolNode, tools_condition  # Did the LLM ask for a tool?
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END 
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages


In [2]:
# create custom tool 
@tool
def calculate_average(scores: list[int]) -> float:
    # add doc string 
    """
      Calculate the average of a list of scores.
        Args:
            scores (list[int]): A list of integer scores.
        Returns:
            float: The average score.
    """
    return sum(scores) / len(scores)





In [3]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [4]:
# create list of tools
tools = [calculate_average]

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)


In [5]:
# Create LLM node 
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [6]:
# create tool node 
tool_node = ToolNode(tools)

In [7]:
# create nodes and edges
graph = StateGraph(State)

graph.add_node("chatbot", chatbot)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chatbot")

graph.add_conditional_edges(
    "chatbot",
    tools_condition,
)

graph.add_edge("tools", "chatbot")

workflow = graph.compile()

In [8]:
result = workflow.invoke(
    {"messages": [HumanMessage(content="What is average of this list of scores: [85, 90, 78, 92, 88]?")]}
)

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What is average of this list of scores: [85, 90, 78, 92, 88]?
================================== Ai Message ==================================
Tool Calls:
  calculate_average (call_Z6CBVRvDpZuu1VrQ6ZGeWz5Q)
 Call ID: call_Z6CBVRvDpZuu1VrQ6ZGeWz5Q
  Args:
    scores: [85, 90, 78, 92, 88]
================================= Tool Message =================================
Name: calculate_average

86.6
================================== Ai Message ==================================

The average of the list of scores [85, 90, 78, 92, 88] is 86.6.
